# Proyecto 1 — Monitoreo transaccional: detectar lo que el orden revela

**Universidad del Valle · Deep Learning 2026 · Kevin Recinos**

Integrantes: _(pendiente)_

**Pregunta central.** ¿El orden de las transacciones aporta información que las variables
agregadas no capturan, bajo qué condiciones y cuánto vale esa información en quetzales?

**Ruta de datos elegida: A — generador sintético propio.** Se eligió sobre la Ruta B porque
permite controlar qué mecanismo de fraude depende del orden y cuál no. Ese control es lo que
convierte la comparación A vs B en evidencia: si el modelo secuencial gana solo donde el orden
importa por construcción, la conclusión es verificable y no una correlación afortunada.

---

### Mapa del notebook

| Bloque | Contenido | Evidencia del informe |
|---|---|---|
| 0 | Configuración y entorno | Reproducibilidad |
| 1 | Generador, secuencias, partición temporal, antifuga | 1 — Integridad de datos |
| 2 | Modelo A (sin orden) y Modelo B (secuencial) | 2 — Comparación común |
| 3 | Permutación controlada y segunda prueba | 3 — Valor del orden |
| 4 | Apuesta C: hipótesis, control, veredicto | 4 — Apuesta del equipo |
| 5 | Umbral, costo y recomendación | 5 y 6 — Decisión y límites |

## Bloque 0 — Configuración y entorno

Una sola semilla gobierna generación de datos, partición, inicialización de pesos y barajado
de lotes. Las constantes de diseño viven en `src/config.py` y se declaran **antes** de ver
cualquier dato; el notebook no las redefine.

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore", category=FutureWarning)

from src import config
from src.utils import fijar_semillas, hash_df, dispositivo, asegurar_directorios, resumen_entorno

asegurar_directorios()
SEMILLA = fijar_semillas()
DISPOSITIVO = dispositivo()

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 9})

print(f"semilla global   : {SEMILLA}")
print(f"dispositivo      : {DISPOSITIVO}")
print(f"metrica principal: {config.METRICA_PRINCIPAL}  (la exactitud NO se reporta como principal)")
print(f"costos           : FN = Q{config.COSTO_FN:,}  |  FP = Q{config.COSTO_FP:,}")

semilla global   : 2026
dispositivo      : cpu
metrica principal: auc_pr  (la exactitud NO se reporta como principal)
costos           : FN = Q4,200  |  FP = Q180


In [2]:
# Versiones exactas del entorno de ejecucion (van tambien al README).
resumen_entorno()

,componente,version
0,python,3.11.9
1,plataforma,Windows-10-10.0.26200-SP0
2,numpy,2.4.6
3,pandas,2.3.3
4,sklearn,1.9.0
5,torch,2.13.0+cpu
6,matplotlib,3.11.0


## Bloque 1 — Datos, secuencias y protocolo temporal

> **Evidencia 1 del informe: integridad de datos.**
> Origen, tamaño, tasa de fraude, construcción de secuencias, partición temporal y
> controles contra fuga de información.

### 1.1 Generador sintético — los tres mecanismos de fraude  *(T2)*

| Mecanismo | Patrón en palabras sencillas | ¿Depende del orden? |
|---|---|---|
| F1 — escalada | Varias compras pequeñas de prueba y, enseguida, una compra grande. | **Sí.** Los mismos montos en otro orden son inofensivos. |
| F2 — ráfaga | Muchas transacciones en pocos minutos, en comercios distintos. | Parcialmente: importa la densidad temporal más que la secuencia exacta. |
| F3 — atípico aislado | Una sola transacción de monto muy raro para esa tarjeta. | **No.** Es el control: la línea base A debería detectarlo tan bien como B. |

F3 existe a propósito. Sin un mecanismo que *no* dependa del orden, cualquier ventaja del
modelo secuencial sería imposible de atribuir.

In [3]:
# TODO (T2): generador sintetico reproducible -> src/generador.py

### 1.2 Construcción de secuencias  *(T3)*

Cada ejemplo es la historia de una tarjeta hasta el evento actual **inclusive**, y se predice
si ese evento es fraudulento. Ese es el horizonte que enfrenta un motor antifraude real:
autorizar o bloquear la transacción que está ocurriendo ahora.

In [4]:
# TODO (T3): construccion de secuencias y particion temporal

### 1.3 Partición temporal y controles antifuga  *(T3, T4)*

Lo más antiguo entrena, lo intermedio valida, lo más reciente prueba. **El conjunto de prueba
se mira una sola vez**, después de fijar arquitectura, hiperparámetros y umbral.

In [5]:
# TODO (T4): agregadas causales + escalado ajustado SOLO en train

## Bloque 2 — Núcleo comparable: A contra B

> **Evidencia 2 del informe: comparación común.**
> Mismos datos, misma partición, mismo horizonte. AUC-PR y, en el umbral elegido,
> precisión, exhaustividad y F1.

### 2.1 Modelo A — línea base sin orden  *(T5)*

In [6]:
# TODO (T5): modelo A sobre variables agregadas

### 2.2 Modelo B — modelo secuencial  *(T6)*

In [7]:
# TODO (T6): modelo B sobre eventos ordenados

### 2.3 Comparación A vs B  *(T7)*

In [8]:
# TODO (T7): curva PR conjunta, AUC-PR, y precision/recall/F1 en el umbral de validacion

## Bloque 3 — ¿De verdad usó el orden?

> **Evidencia 3 del informe: valor del orden.**
> Una mejora de métricas no demuestra por sí sola que el modelo leyó la secuencia.
> Estas dos pruebas intentan refutar nuestra propia conclusión.

### 3.1 Prueba obligatoria — permutación controlada  *(T8)*

In [9]:
# TODO (T8): barajar el orden DENTRO de cada secuencia, sin tocar eventos ni agregadas

### 3.2 Segunda prueba — desempeño por mecanismo de fraude  *(T9)*

In [10]:
# TODO (T9): AUC-PR de A y B desglosado por F1 / F2 / F3

## Bloque 4 — Apuesta del equipo

> **Evidencia 4 del informe: apuesta del equipo.**
> Hipótesis previa, control experimental, métrica de éxito y veredicto — aunque falle.

**La hipótesis se escribe y se commitea ANTES de entrenar C y ANTES de tocar el conjunto de prueba.**

### 4.1 Hipótesis previa  *(T10)*

> Creemos que ______ mejorará ______ porque ______. Lo consideraremos útil si ______.

_(pendiente — T10)_

In [11]:
# TODO (T11): entrenar C y su control, y emitir veredicto explicito

## Bloque 5 — Umbral, costo y recomendación

> **Evidencias 5 y 6 del informe: decisión económica, recomendación y límites.**
> Un fraude no detectado cuesta Q4,200; bloquear una transacción legítima cuesta Q180.
> El umbral se elige barriendo **validación** y se aplica una sola vez a prueba.

In [12]:
# TODO (T12): barrido de umbral por costo esperado y proyeccion mensual

## Matriz de evidencias

| Evidencia | Figura o tabla | Conclusión | Limitación |
|---|---|---|---|
| 1 — Integridad de datos | _(pendiente)_ | | |
| 2 — Comparación común A vs B | _(pendiente)_ | | |
| 3 — Valor del orden | _(pendiente)_ | | |
| 4 — Apuesta del equipo | _(pendiente)_ | | |
| 5 — Decisión económica | _(pendiente)_ | | |
| 6 — Recomendación y límites | _(pendiente)_ | | |